# Chapter 13 — Reductions II: Shared Memory & Block Reductions

> Course: **llm.c — Zero to Hero**, Chapter 13 of ~20.
> Builds on Chapter 12 (warp shuffles).

In Chapter 12 we summed 32 values in a warp using shuffles. But GPT-2 small has `C = 768` channels per row. To compute the LayerNorm mean over 768 elements, we need a **block-level reduction** — combining results from multiple warps within a thread block.

The classical pattern uses **shared memory** (`__shared__`): an on-chip scratchpad accessible to all threads in a block. We'll combine warp shuffles + shared memory to reduce up to 1024 values per block.

### Learning objectives

By the end of this chapter you will:

- Use `__shared__` to share data between threads in a block.
- Synchronize threads with `__syncthreads()`.
- Implement `blockReduceSum` as a 2-stage warp → shared mem → warp reduction.
- Use it to write a per-row LayerNorm forward kernel that handles `C` of any size up to 1024.


## 1. Concept — Shared Memory

Each thread block gets a small, fast on-chip memory called **shared memory** — typically 48 KB to 100 KB depending on the GPU. It's:

- **Visible only to threads in the same block.** Not shared across blocks.
- **As fast as L1 cache** — ~100× faster than global memory.
- **Allocated in the kernel** with `__shared__`.

Declaration:

```c
__global__ void kernel(...) {
    __shared__ float scratch[32];   // 32 floats, shared by all threads in this block
    int t = threadIdx.x;
    scratch[t] = something;
    __syncthreads();                // wait until ALL threads have written
    // now every thread can read every other thread's `scratch[*]` safely
}
```

The crucial bit is **`__syncthreads()`** — a barrier that all threads in the block must reach before any can continue. Without it, one thread might read `scratch[5]` before thread 5 has written to it, giving you garbage. Forgetting `__syncthreads()` is the single most common source of CUDA bugs.

### Why shared memory + warps for block reductions?

Warps shuffle within themselves for free. Across warps, you can't shuffle — you have to use shared memory. The standard pattern:

1. **Stage 1** — each warp does a warp-level reduction. Now lane 0 of each warp holds the warp's partial sum.
2. **Stage 2** — those `numWarps` partial sums are written to shared memory.
3. **Stage 3** — one warp reads them back and does a final warp reduction.

For a 1024-thread block (= 32 warps), that's 32 partial sums in shared mem, then one more warp reduction = 5+5 = 10 instructions of "real" reduction work, plus the shared-mem traffic.


## 2. `blockReduceSum` in Code

Here's the standard implementation (essentially what `llm.c`'s `cuda_utils.cuh` provides):

```c
__device__ float warpReduceSum(float val) {
    for (int offset = 16; offset > 0; offset /= 2)
        val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}

__device__ float blockReduceSum(float val) {
    __shared__ float warp_sums[32];          // up to 32 warps per block
    int lane    = threadIdx.x & 31;          // lane within warp (0..31)
    int warp_id = threadIdx.x >> 5;          // warp id within block (0..31)
    int n_warps = (blockDim.x + 31) / 32;

    // Stage 1: per-warp reduction
    val = warpReduceSum(val);

    // Stage 2: lane 0 of each warp writes its warp sum to shared mem
    if (lane == 0) warp_sums[warp_id] = val;
    __syncthreads();

    // Stage 3: first warp loads the warp sums, reduces them
    val = (threadIdx.x < n_warps) ? warp_sums[threadIdx.x] : 0.0f;
    if (warp_id == 0) val = warpReduceSum(val);
    return val;     // thread 0 holds the block sum
}
```

The body is **15 lines**. It works for any `blockDim.x` from 32 to 1024, returning the full block sum to thread 0.

If you want every thread to know the block sum (common need), broadcast via shared memory after step 3:

```c
__shared__ float bcast;
if (threadIdx.x == 0) bcast = val;
__syncthreads();
return bcast;
```


## 3. Demo — `blockReduceSum` Verifying

In [ ]:
!mkdir -p course/ch13_build


In [ ]:
%%writefile course/ch13_build/block_reduce.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float val) {
    for (int offset = 16; offset > 0; offset /= 2)
        val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}

__device__ float blockReduceSum(float val) {
    __shared__ float warp_sums[32];
    int lane    = threadIdx.x & 31;
    int warp_id = threadIdx.x >> 5;
    int n_warps = (blockDim.x + 31) / 32;

    val = warpReduceSum(val);
    if (lane == 0) warp_sums[warp_id] = val;
    __syncthreads();
    val = (threadIdx.x < n_warps) ? warp_sums[threadIdx.x] : 0.0f;
    if (warp_id == 0) val = warpReduceSum(val);
    return val;
}

// One block per row. Each thread loads one element, reduces, writes block sum to out[row].
__global__ void row_sum_kernel(float* out, const float* inp, int N) {
    int row = blockIdx.x;
    int t   = threadIdx.x;
    float v = (t < N) ? inp[row * N + t] : 0.0f;
    v = blockReduceSum(v);
    if (t == 0) out[row] = v;
}

int main(void) {
    int rows = 8, N = 768;
    float* h_inp = (float*) malloc(rows*N*4);
    float* h_out = (float*) malloc(rows*4);
    for (int r = 0; r < rows; r++)
        for (int i = 0; i < N; i++)
            h_inp[r*N + i] = (float)((i + r*7) % 17) / 10.0f;

    float *d_inp, *d_out;
    cudaMalloc(&d_inp, rows*N*4); cudaMalloc(&d_out, rows*4);
    cudaMemcpy(d_inp, h_inp, rows*N*4, cudaMemcpyHostToDevice);

    // Use blockDim = next-power-of-2 above N, capped at 1024
    int block = 1024;
    row_sum_kernel<<<rows, block>>>(d_out, d_inp, N);
    cudaMemcpy(h_out, d_out, rows*4, cudaMemcpyDeviceToHost);

    for (int r = 0; r < rows; r++) {
        double cpu = 0;
        for (int i = 0; i < N; i++) cpu += h_inp[r*N + i];
        printf("row %d: gpu = %.4f  cpu = %.4f  diff = %.2e\n",
               r, h_out[r], (float)cpu, fabsf(h_out[r] - (float)cpu));
    }
    cudaFree(d_inp); cudaFree(d_out);
    free(h_inp); free(h_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13_build/block_reduce course/ch13_build/block_reduce.cu && ./course/ch13_build/block_reduce


All 8 rows should match the CPU sum within float32 noise (~`1e-5`). The block sums **768 values** with 1024 threads, in 2 stages of warp reductions plus 32 shared-memory stores — and we get a single answer in thread 0 in microseconds.


## 4. Demo — LayerNorm Forward on the GPU

Now let's actually use it for LayerNorm. Each block handles one `(b, t)` row of length `C`. Within the block:

1. All threads load their slice of the row.
2. `blockReduceSum` to get the row sum → divide by C → mean.
3. Each thread computes `(x[i] - mean)^2` and we `blockReduceSum` again → variance.
4. Compute `rstd = 1/sqrt(var + eps)`.
5. Each thread writes `out[i] = (x[i] - mean) * rstd * weight[i] + bias[i]`.

This is the structure of `dev/cuda/layernorm_forward.cu`'s mid-tier kernels.


In [ ]:
%%writefile course/ch13_build/layernorm_block.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float val) {
    for (int o = 16; o > 0; o /= 2) val += __shfl_down_sync(0xffffffff, val, o);
    return val;
}

__device__ float blockReduceSum(float val) {
    __shared__ float warp_sums[32];
    int lane    = threadIdx.x & 31;
    int warp_id = threadIdx.x >> 5;
    int n_warps = (blockDim.x + 31) / 32;
    val = warpReduceSum(val);
    if (lane == 0) warp_sums[warp_id] = val;
    __syncthreads();
    val = (threadIdx.x < n_warps) ? warp_sums[threadIdx.x] : 0.0f;
    if (warp_id == 0) val = warpReduceSum(val);
    return val;
}

// One block per (b, t) row. blockDim.x must be a power of 2 between 32 and 1024.
// Inputs: inp (B*T, C), weight (C,), bias (C,). Output: out (B*T, C), mean (B*T,), rstd (B*T,).
__global__ void layernorm_kernel(float* out, float* mean, float* rstd,
                                 const float* inp, const float* weight, const float* bias,
                                 int N, int C) {
    extern __shared__ float bcast[];   // shared scalar broadcast (m, then r)
    int row = blockIdx.x;
    int t   = threadIdx.x;
    const float* x   = inp + row * C;
    float* out_row   = out + row * C;

    // pass 1: mean
    float sum = 0.0f;
    for (int i = t; i < C; i += blockDim.x) sum += x[i];
    sum = blockReduceSum(sum);
    if (t == 0) bcast[0] = sum / C;
    __syncthreads();
    float m = bcast[0];

    // pass 2: variance
    float vs = 0.0f;
    for (int i = t; i < C; i += blockDim.x) { float d = x[i] - m; vs += d*d; }
    vs = blockReduceSum(vs);
    if (t == 0) bcast[1] = 1.0f / sqrtf(vs / C + 1e-5f);
    __syncthreads();
    float r = bcast[1];

    // pass 3: write out (and cache mean, rstd)
    for (int i = t; i < C; i += blockDim.x) out_row[i] = (x[i] - m) * r * weight[i] + bias[i];
    if (t == 0) { mean[row] = m; rstd[row] = r; }
}

int main(void) {
    int B = 4, T = 8, C = 768;
    int N = B * T;
    float* h_inp = (float*) malloc(N*C*4);
    float* h_w   = (float*) malloc(C*4);
    float* h_b   = (float*) malloc(C*4);
    float* h_gpu = (float*) malloc(N*C*4);
    float* h_cpu = (float*) malloc(N*C*4);
    for (int i = 0; i < N*C; i++) h_inp[i] = (float)((i*13) % 31) / 10.0f - 1.5f;
    for (int i = 0; i < C; i++) { h_w[i] = 1.0f + (i%5)*0.1f; h_b[i] = (float)((i*7)%9)/10.0f; }

    // CPU reference
    for (int r = 0; r < N; r++) {
        float m = 0; for (int i = 0; i < C; i++) m += h_inp[r*C+i]; m /= C;
        float v = 0; for (int i = 0; i < C; i++) { float d = h_inp[r*C+i]-m; v += d*d; } v /= C;
        float s = 1.0f / sqrtf(v + 1e-5f);
        for (int i = 0; i < C; i++) h_cpu[r*C+i] = (h_inp[r*C+i] - m) * s * h_w[i] + h_b[i];
    }

    float *d_inp, *d_w, *d_b, *d_out, *d_mean, *d_rstd;
    cudaMalloc(&d_inp, N*C*4); cudaMalloc(&d_w, C*4); cudaMalloc(&d_b, C*4);
    cudaMalloc(&d_out, N*C*4); cudaMalloc(&d_mean, N*4); cudaMalloc(&d_rstd, N*4);
    cudaMemcpy(d_inp, h_inp, N*C*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_w,   h_w,   C*4,   cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,   h_b,   C*4,   cudaMemcpyHostToDevice);

    int block = 256;
    layernorm_kernel<<<N, block, 2*sizeof(float)>>>(d_out, d_mean, d_rstd, d_inp, d_w, d_b, N, C);

    cudaMemcpy(h_gpu, d_out, N*C*4, cudaMemcpyDeviceToHost);
    float maxerr = 0;
    for (int i = 0; i < N*C; i++) { float e = fabsf(h_gpu[i] - h_cpu[i]); if (e > maxerr) maxerr = e; }
    printf("LayerNorm GPU max diff vs CPU: %.2e\n", maxerr);

    cudaFree(d_inp); cudaFree(d_w); cudaFree(d_b); cudaFree(d_out); cudaFree(d_mean); cudaFree(d_rstd);
    free(h_inp); free(h_w); free(h_b); free(h_gpu); free(h_cpu);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13_build/layernorm_block course/ch13_build/layernorm_block.cu && ./course/ch13_build/layernorm_block


You should see `~1e-5` agreement between GPU and CPU LayerNorm. **You just wrote a working LayerNorm forward on the GPU.** Compared to the CPU version (Chapter 3), the structure is the same: 3 passes over each row (mean → variance → write). Difference: instead of a `for` loop over `C`, all `blockDim.x` threads cooperate via warp+block reductions.

This is the structure of `llmc/layernorm.cuh::layernorm_forward_kernel6` — except that one uses Packed128 vectorized loads and bf16 weights/bias caching for extra speed. The reduction logic is identical.


## 5. Translation Bridge

| CPU | GPU per-block reduction |
|---|---|
| `for (i) sum += x[i];` | `for (i = t; i < C; i += blockDim.x) sum += x[i]; sum = blockReduceSum(sum);` |
| `m = sum / C` | `if (t == 0) bcast[0] = sum / C; __syncthreads(); m = bcast[0];` |
| Sequential | All `blockDim.x` threads work together; final answer in thread 0 |
| ~`O(C)` ops sequentially | ~`O(C / blockDim.x)` ops per thread, plus log-depth reduction |

The `__shared__` array lets thread-0 broadcast the mean to other threads in pass 2 and 3. **One scalar in shared mem** is enough — no need to allocate big buffers.


## 6. TODO Exercise — Block Max

In [ ]:
%%writefile course/ch13_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ float warpReduceMax(float val) {
    // TODO: butterfly reduce with fmaxf (5 shuffles)
    return val;
}

__device__ float blockReduceMax(float val) {
    // TODO: warp reduce, then store warp maxes in shared mem, then reduce again with first warp
    return val;
}

__global__ void row_max_kernel(float* out, const float* inp, int C) {
    int row = blockIdx.x;
    int t   = threadIdx.x;
    float v = -1e30f;
    for (int i = t; i < C; i += blockDim.x) v = fmaxf(v, inp[row*C + i]);
    v = blockReduceMax(v);
    if (t == 0) out[row] = v;
}

int main(void) {
    int rows = 4, C = 1000;
    float* h_inp = (float*) malloc(rows*C*4);
    float* h_out = (float*) malloc(rows*4);
    for (int r = 0; r < rows; r++)
        for (int i = 0; i < C; i++)
            h_inp[r*C + i] = (float)((i + r*100) % 997) - 500.0f;
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, rows*C*4); cudaMalloc(&d_out, rows*4);
    cudaMemcpy(d_inp, h_inp, rows*C*4, cudaMemcpyHostToDevice);
    row_max_kernel<<<rows, 256>>>(d_out, d_inp, C);
    cudaMemcpy(h_out, d_out, rows*4, cudaMemcpyDeviceToHost);
    int ok = 1;
    for (int r = 0; r < rows; r++) {
        float m = -1e30f;
        for (int i = 0; i < C; i++) if (h_inp[r*C+i] > m) m = h_inp[r*C+i];
        if (h_out[r] != m) { ok = 0; printf("row %d: gpu=%.0f cpu=%.0f\n", r, h_out[r], m); }
    }
    printf("%s\n", ok ? "PASS" : "FAIL");
    cudaFree(d_inp); cudaFree(d_out); free(h_inp); free(h_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13_build/exercise1 course/ch13_build/exercise1.cu && ./course/ch13_build/exercise1


### Solution

In [ ]:
%%writefile course/ch13_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ float warpReduceMax(float val) {
    for (int o = 16; o > 0; o /= 2) val = fmaxf(val, __shfl_down_sync(0xffffffff, val, o));
    return val;
}

__device__ float blockReduceMax(float val) {
    __shared__ float warp_max[32];
    int lane = threadIdx.x & 31;
    int wid  = threadIdx.x >> 5;
    int nw   = (blockDim.x + 31) / 32;
    val = warpReduceMax(val);
    if (lane == 0) warp_max[wid] = val;
    __syncthreads();
    val = (threadIdx.x < nw) ? warp_max[threadIdx.x] : -1e30f;
    if (wid == 0) val = warpReduceMax(val);
    return val;
}

__global__ void row_max_kernel(float* out, const float* inp, int C) {
    int row = blockIdx.x;
    int t   = threadIdx.x;
    float v = -1e30f;
    for (int i = t; i < C; i += blockDim.x) v = fmaxf(v, inp[row*C + i]);
    v = blockReduceMax(v);
    if (t == 0) out[row] = v;
}

int main(void) {
    int rows = 4, C = 1000;
    float* h_inp = (float*) malloc(rows*C*4);
    float* h_out = (float*) malloc(rows*4);
    for (int r = 0; r < rows; r++)
        for (int i = 0; i < C; i++)
            h_inp[r*C + i] = (float)((i + r*100) % 997) - 500.0f;
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, rows*C*4); cudaMalloc(&d_out, rows*4);
    cudaMemcpy(d_inp, h_inp, rows*C*4, cudaMemcpyHostToDevice);
    row_max_kernel<<<rows, 256>>>(d_out, d_inp, C);
    cudaMemcpy(h_out, d_out, rows*4, cudaMemcpyDeviceToHost);
    int ok = 1;
    for (int r = 0; r < rows; r++) {
        float m = -1e30f;
        for (int i = 0; i < C; i++) if (h_inp[r*C+i] > m) m = h_inp[r*C+i];
        if (h_out[r] != m) ok = 0;
    }
    printf("%s\n", ok ? "PASS" : "FAIL");
    cudaFree(d_inp); cudaFree(d_out); free(h_inp); free(h_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13_build/exercise1_sol course/ch13_build/exercise1_sol.cu && ./course/ch13_build/exercise1_sol


## Recap

You now know:

- **Shared memory** (`__shared__`) is a fast scratchpad visible to all threads in a block.
- **`__syncthreads()`** is a barrier — all threads in the block must reach it before any continue.
- **`blockReduceSum`** = warp reduce → write to `__shared__[warp_id]` → `__syncthreads()` → first warp reads and warp-reduces again. 15-line pattern.
- **LayerNorm forward on GPU**: one block per row, three passes (mean → variance → write), each pass uses one `blockReduceSum`.

### What's next

**Chapter 14 — cuBLAS for matmul.** Matmul is too important and complex to hand-roll well. NVIDIA's cuBLAS library has spent decades being optimized — *we should use it*. We'll see how `llmc/matmul.cuh` calls `cublasLtMatmul`, including the *fused epilogue* (matmul + bias + GELU in one kernel).

When you're ready, say **"proceed to Chapter 14"**.
